# FYP Week 6 — Hyperparameter Tuning

**Research question:** Can systematic hyperparameter selection improve generalization in Malaysian salary prediction?

**This notebook's scope:** Verify the Week 5 data setup, reproduce the Decision Tree baseline with 5-fold CV, tune its complexity using training-only Grid Search, and save results. **Do not use the held-out test set for tuning or model selection.**

- Target: G8_Total_salaries_and_wages only; G14 is excluded as both target and feature.
- Fixed split: 80% development/training, 20% held-out test; random_state=42.
- Same occupation/industry recoding and preprocessing as Week 5, fitted within CV folds.
- Main selection metric: CV MAE (RM); RMSE/R² and Train–CV gap provide context.
- **Important:** Week 5's test scores have already been inspected. The next stage will use a more rigorous validation design (e.g. nested CV) for model comparisons; the minimum tuning CV score alone is not unbiased final performance.
- This notebook is independent of Week 5 variables. It needs the same local fulldata.csv; raw personal/salary data are not uploaded here. Do not commit executed outputs containing confidential row-level data.

**为什么做：** Week 5 restricted Decision Tree had approximately RM 898.70 CV MAE with max_depth=10 and min_samples_leaf=10, but those values were not systematically searched. We now test whether another complexity setting performs better under the same training-only CV protocol.

Source: Week 5 — https://github.com/Kyle050206/FYP/blob/main/week5_analysis.ipynb


## 1. Environment & data location

**What / Why:** Record versions and locate the original CSV. Keep the source file unchanged. When PyCharm's working directory differs, set DATA_PATH manually to its exact location.


In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone

SEED = 42
TARGET = "G8_Total_salaries_and_wages"
OUTPUT_DIR = Path("week6_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Override if needed, e.g. DATA_PATH = Path("/Users/.../data/fulldata.csv")
DATA_PATH = None
if DATA_PATH is None:
    candidates = [
        Path("../data/fulldata.csv"),   # Same relative path used in Week 5
        Path("data/fulldata.csv"),
        Path("fulldata.csv"),
        Path("../fulldata.csv"),
    ]
    matches = [p for p in candidates if p.is_file()]
    if not matches:
        raise FileNotFoundError(
            f"Cannot find fulldata.csv. Current working directory: {Path.cwd()}. "
            "Set DATA_PATH above to the exact location of your original CSV."
        )
    DATA_PATH = matches[0]

print("Working directory:", Path.cwd())
print("Data source:", DATA_PATH.resolve())
print("Python:", platform.python_version(), "| sklearn:", sklearn.__version__)
print("Output folder:", OUTPUT_DIR.resolve())

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("G8 missing:", df[TARGET].isna().sum())


## 2. Reproduce Week 5 feature preparation

**Why:** A fair tuning experiment must use the same recoding, nine input features and target definition as Week 5. Occupation/industry lookups are fixed code mappings, not statistics learned from test data.


In [ ]:
df_recoded = df.drop(columns=[
    "Field_of_Study",
    "Highest_Certificate_Obtained",
    "Citizenship",
    "G14_Total_salaries_and_wages_includes_overtime",
]).copy()

occupation_mapping = {
    "0": "Armed Forces",
    "1": "Managers",
    "2": "Professionals",
    "3": "Technicians and Associate Professionals",
    "4": "Clerical Support Workers",
    "5": "Service and Sales Workers",
    "6": "Skilled Agricultural Workers",
    "7": "Craft and Related Trades Workers",
    "8": "Plant and Machine Operators",
    "9": "Elementary Occupations",
}

occupation_codes = (
    df_recoded["Occupation_2D"].astype(int).astype(str).str.zfill(2)
)
df_recoded["Occupation_Group"] = (
    occupation_codes.str[0].map(occupation_mapping).astype("category")
)

industry_ranges = {
    "A": [1, 2, 3], "B": list(range(5, 10)),
    "C": list(range(10, 34)), "D": [35],
    "E": [36, 37, 38, 39], "F": [41, 42, 43],
    "G": [45, 46, 47], "H": [49, 50, 51, 52, 53],
    "I": [55, 56], "J": list(range(58, 64)),
    "K": [64, 65, 66], "L": [68],
    "M": list(range(69, 76)), "N": list(range(77, 83)),
    "O": [84], "P": [85], "Q": [86, 87, 88],
    "R": [90, 91, 92, 93], "S": [94, 95, 96],
    "T": [97, 98], "U": [99],
}
industry_mapping = {
    str(code).zfill(2): section
    for section, codes in industry_ranges.items()
    for code in codes
}
industry_codes = (
    df_recoded["Industry_2D"].astype(int).astype(str).str.zfill(2)
)
df_recoded["Industry_Section"] = (
    industry_codes.map(industry_mapping).astype("category")
)

FEATURES = [
    "State", "Strata", "Sex", "Age", "Ethnic_Group",
    "Marital_Status", "Educational_Attainment",
    "Occupation_Group", "Industry_Section",
]
NOMINAL = [
    "State", "Strata", "Sex", "Ethnic_Group",
    "Marital_Status", "Occupation_Group", "Industry_Section",
]

X = df_recoded[FEATURES].copy()
y = df_recoded[TARGET].copy()

for column in NOMINAL:
    X[column] = X[column].astype("category")
X["Educational_Attainment"] = pd.Categorical(
    X["Educational_Attainment"],
    categories=[1, 2, 3, 4],
    ordered=True,
)

if X.isna().any().any() or y.isna().any():
    raise ValueError("Missing values found. Re-check input CSV and recoding before modelling.")
if not np.isfinite(y).all() or not (y > 0).all():
    raise ValueError("G8 must be finite and positive (needed for later log-G8 analysis).")
assert "G14_Total_salaries_and_wages_includes_overtime" not in X.columns

print("Occupation groups:", X["Occupation_Group"].nunique(),
      "| Unmapped:", X["Occupation_Group"].isna().sum())
print("Industry sections:", X["Industry_Section"].nunique(),
      "| Unmapped:", X["Industry_Section"].isna().sum())
print("Number of raw features:", len(FEATURES))
display(X.dtypes.to_frame("dtype"))


## 3. Freeze the 80/20 split and build preprocessing

**Why:** Hyperparameter search happens only inside the original 80% training portion. Keep the 20% test portion aside. The pipeline refits its encoder for every CV training fold, preventing preprocessing leakage.

This is the same random_state and feature engineering used in Week 5. If the CSV is unchanged, expected sizes are 21,940 training and 5,485 testing records.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

preprocessor = ColumnTransformer([
    ("nominal", OneHotEncoder(handle_unknown="ignore"), NOMINAL),
    ("ordinal", OrdinalEncoder(categories=[[1, 2, 3, 4]]),
     ["Educational_Attainment"]),
    ("numerical", "passthrough", ["Age"]),
])

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

def dt_pipeline(model):
    return Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", model),
    ])

print("Training:", X_train.shape, "| Held-out test:", X_test.shape)
print("Training labels:", y_train.shape, "| Held-out test labels:", y_test.shape)
if (len(X_train), len(X_test)) != (21940, 5485):
    print("NOTE: Counts differ from Week 5; confirm fulldata.csv is the same version.")

# No fit / predict / score against X_test or y_test in this notebook section.


## 4. Reproduce baseline Decision Tree results (5-fold CV)

**Why:** Before tuning we must establish a reference using exactly the same folds and preprocessing. Compare the prior restricted tree (depth=10, leaf=10) against an unrestricted tree; focus on Train MAE versus CV MAE.

Week 5 reference: Restricted DT CV MAE ≈ RM 898.70; Unrestricted DT CV MAE ≈ RM 1,032.69. These are historical outputs, not Week 6 measurements.


In [ ]:
BASELINE_MODELS = {
    "Unrestricted DT": DecisionTreeRegressor(random_state=SEED),
    "Restricted DT (Week 5)": DecisionTreeRegressor(
        max_depth=10, min_samples_leaf=10, random_state=SEED
    ),
}

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

baseline_rows = []
for name, model in BASELINE_MODELS.items():
    scores = cross_validate(
        dt_pipeline(model),
        X_train, y_train,
        cv=cv, scoring=scoring,
        return_train_score=True,
        n_jobs=1,
        error_score="raise",
    )
    baseline_rows.append({
        "Model": name,
        "Train MAE": -scores["train_MAE"].mean(),
        "CV MAE": -scores["test_MAE"].mean(),
        "CV MAE SD": scores["test_MAE"].std(),
        "Train RMSE": -scores["train_RMSE"].mean(),
        "CV RMSE": -scores["test_RMSE"].mean(),
        "Train R2": scores["train_R2"].mean(),
        "CV R2": scores["test_R2"].mean(),
    })

baseline_table = pd.DataFrame(baseline_rows)
baseline_table["MAE Gap"] = baseline_table["CV MAE"] - baseline_table["Train MAE"]
display(baseline_table.round(3))
baseline_table.to_csv(OUTPUT_DIR / "decision_tree_baseline_cv.csv", index=False)

week5_restricted_cv_mae = 898.6962632497518
week6_restricted_cv_mae = baseline_table.loc[
    baseline_table["Model"] == "Restricted DT (Week 5)", "CV MAE"
].iloc[0]
print("Week 5 restricted CV MAE:", round(week5_restricted_cv_mae, 2))
print("Reproduced restricted CV MAE:", round(week6_restricted_cv_mae, 2))
print("Difference (RM):", round(week6_restricted_cv_mae - week5_restricted_cv_mae, 3))


## 5. Decision Tree Grid Search on the training portion

**Question:** Are depth=10 and min_samples_leaf=10 good settings, or can a different combination lower CV MAE?

**Search design:** 7 candidate depths × 4 minimum leaf sizes = 28 combinations; 5-fold CV = 140 CV fits. All other model settings stay at their defaults (except random_state). The previous (10,10) configuration is included as a candidate.

**Important:** This is an initial search range, not a guarantee of global optimality. It does not use the held-out test set.


In [ ]:
param_grid = {
    "model__max_depth": [4, 6, 8, 10, 12, 16, None],
    "model__min_samples_leaf": [5, 10, 20, 40],
}

grid_search = GridSearchCV(
    estimator=dt_pipeline(DecisionTreeRegressor(random_state=SEED)),
    param_grid=param_grid,
    scoring=scoring,
    refit="MAE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    error_score="raise",
)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_cv_mae = -grid_search.best_score_
best_dt_pipeline = grid_search.best_estimator_  # Refitted on the full training portion

print("Best candidate hyperparameters:", best_params)
print(f"Best tuning CV MAE: RM {best_cv_mae:,.2f}")
print("Selected model (training-fitted):")
display(best_dt_pipeline.named_steps["model"])


## 6. Inspect all candidate results & assess overfitting

**Why:** The best mean CV MAE is the selection criterion; the train–validation gap and across-fold variation help interpret model complexity. Avoid claiming the winning CV score is an independent final test result.


In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_).copy()
grid_results["max_depth"] = grid_results["param_model__max_depth"].astype(str)
grid_results["min_samples_leaf"] = grid_results["param_model__min_samples_leaf"].astype(int)
grid_results["Train MAE"] = -grid_results["mean_train_MAE"]
grid_results["CV MAE"] = -grid_results["mean_test_MAE"]
grid_results["CV MAE SD"] = grid_results["std_test_MAE"]
grid_results["Train RMSE"] = -grid_results["mean_train_RMSE"]
grid_results["CV RMSE"] = -grid_results["mean_test_RMSE"]
grid_results["CV R2"] = grid_results["mean_test_R2"]
grid_results["MAE Gap"] = grid_results["CV MAE"] - grid_results["Train MAE"]
grid_results["Rank"] = grid_results["rank_test_MAE"]

columns = [
    "max_depth", "min_samples_leaf", "Train MAE", "CV MAE",
    "CV MAE SD", "Train RMSE", "CV RMSE", "CV R2", "MAE Gap", "Rank"
]
ranking = grid_results[columns].sort_values(["CV MAE", "Rank"]).reset_index(drop=True)
display(ranking.head(10).round(3))
ranking.to_csv(OUTPUT_DIR / "decision_tree_gridsearch_all.csv", index=False)

baseline_mae = float(week6_restricted_cv_mae)
summary = pd.DataFrame([
    {"Model": "Week 5 restricted DT", "CV MAE (RM)": baseline_mae,
     "Selection": "depth=10, leaf=10"},
    {"Model": "Tuned DT (selection CV)", "CV MAE (RM)": best_cv_mae,
     "Selection": str(best_params)},
])
summary["Improvement vs Week 5 (RM)"] = baseline_mae - summary["CV MAE (RM)"]
display(summary.round(3))
summary.to_csv(OUTPUT_DIR / "decision_tree_tuning_summary.csv", index=False)

print("Caution: tuning CV scores are used to select hyperparameters.")
print("For a less selection-biased performance estimate, use outer/nested CV later.")


## 7. Visualize CV MAE vs. tree complexity

**Problem solved:** See whether deeper trees or smaller leaves consistently help or hurt validation performance instead of relying only on a ranked table.


In [ ]:
depth_order = ["4", "6", "8", "10", "12", "16", "None"]
fig, ax = plt.subplots(figsize=(10, 5))
for leaf in sorted(grid_results["min_samples_leaf"].unique()):
    sub = grid_results.loc[
        grid_results["min_samples_leaf"] == leaf
    ].set_index("max_depth").reindex(depth_order)
    ax.plot(depth_order, sub["CV MAE"], marker="o",
            label=f"min_samples_leaf={leaf}")
ax.set_xlabel("max_depth (None = unlimited)")
ax.set_ylabel("5-Fold CV MAE (RM, lower is better)")
ax.set_title("Decision Tree: validation error across hyperparameters")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "decision_tree_cv_mae_by_depth.png",
            dpi=200, bbox_inches="tight")
plt.show()


## 8. Save experiment metadata and next steps

**What this stage can conclude:** Whether the parameter search found a lower tuning CV MAE than our Week 5 restricted-tree baseline. This is **not yet** final model selection.

**Next Week 6 tasks (not implemented in this notebook yet):**
1. Check the best candidate and compare train/CV gap.
2. Tune shortlisted strong models from Week 5 (CatBoost, LightGBM, etc.) using comparable validation conditions and appropriate computational budgets.
3. Compare original G8 and ln(G8) models on the **same RM scale**.
4. Use an unbiased outer validation procedure / nested CV for selection-performance assessment; be transparent that the previous test set was already inspected.
5. Only after deciding the final model: XAI (SHAP).

No held-out test predictions or test metrics have been computed here.


In [ ]:
metadata = {
    "week": 6,
    "stage": "Decision Tree hyperparameter tuning",
    "data_filename": DATA_PATH.name,
    "n_rows": len(df),
    "n_training": len(X_train),
    "n_heldout_test": len(X_test),
    "target": TARGET,
    "features": FEATURES,
    "test_size": 0.2,
    "random_state": SEED,
    "cv": "5-Fold shuffled KFold",
    "selection_metric": "MAE on original RM scale",
    "parameter_grid": {
        "max_depth": ["4", "6", "8", "10", "12", "16", "None"],
        "min_samples_leaf": [5, 10, 20, 40],
    },
    "best_hyperparameters": {
        "max_depth": best_params["model__max_depth"],
        "min_samples_leaf": best_params["model__min_samples_leaf"],
    },
    "best_tuning_cv_mae_rm": float(best_cv_mae),
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "heldout_test_used_for_tuning": False,
}
with open(OUTPUT_DIR / "decision_tree_experiment_metadata.json",
          "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved:")
for p in sorted(OUTPUT_DIR.glob("decision_tree_*")):
    print(" -", p)

print("\nNext: interpret real tuning results, then validate other candidate models.")
